# 1. Kết nối Google Drive & Cài đặt thư viện
Khi chạy ô này, Colab sẽ xin phép truy cập Google Drive của bạn để lưu lại Model AI (~4GB). Lần sau có đứt mạng chạy lại sẽ không phải tải lại nữa!

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# Chỉ định thư mục lưu Model trên Google Drive
os.environ['HF_HOME'] = '/content/drive/MyDrive/HuggingFace_Cache'

!pip install --upgrade pip
!pip install "mineru[all]>=3.1.3"
!pip install fastapi uvicorn pyngrok nest-asyncio


# 2. Khởi tạo API Server của MinerU trên Colab

In [ ]:
server_code = """
import io
import traceback
import torch
from fastapi import FastAPI, HTTPException
import uvicorn
from pydantic import BaseModel
from PIL import Image
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from mineru_vl_utils import MinerUClient
import os
import base64

# Thiết lập biến môi trường để thư viện AI tự biết tìm Model ở Google Drive
os.environ['HF_HOME'] = '/content/drive/MyDrive/HuggingFace_Cache'

app = FastAPI(title="MinerU PDF Analyzer API on Colab")

model_name = "opendatalab/MinerU2.5-Pro-2604-1.2B"
global_client = None
is_cancelled = False

class AnalyzeRequest(BaseModel):
    file_path: str = ""
    file_base64: str = None

@app.post("/cancel")
def cancel_analysis():
    global is_cancelled
    is_cancelled = True
    return {"status": "success", "message": "Đã yêu cầu hủy"}

@app.post("/load_model")
def load_model():
    global global_client
    if global_client is not None:
        return {"status": "success", "message": "Model đã được tải sẵn trong RAM!"}
    
    try:
        print("Tải/Kiểm tra model từ Google Drive và khởi tạo trên GPU...", flush=True)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        processor = AutoProcessor.from_pretrained(model_name)
        model = Qwen2VLForConditionalGeneration.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16,
            device_map=device
        )
        global_client = MinerUClient(model=model, processor=processor)
        return {"status": "success", "message": "Tải mô hình thành công!"}
    except Exception as e:
        return {"status": "error", "message": str(e)}

@app.post("/analyze")
def analyze_pdf(req: AnalyzeRequest):
    global global_client, is_cancelled
    is_cancelled = False
    
    if global_client is None:
        return {"status": "error", "message": "Vui lòng gọi /load_model trước"}
    
    try:
        print(f"Nhận file: {req.file_path}", flush=True)
        if not req.file_base64:
            return {"status": "error", "message": "Không có dữ liệu Base64"}
            
        file_data = base64.b64decode(req.file_base64)
        local_path = os.path.join("/tmp", req.file_path or "temp.pdf")
        with open(local_path, "wb") as f:
            f.write(file_data)
            
        import fitz
        doc = fitz.open(local_path)
        full_data = []
        
        def extract_style_from_fitz(mineru_bbox, fitz_page):
            page_dict = fitz_page.get_text("dict")
            page_width = fitz_page.rect.width
            page_height = fitz_page.rect.height
            mx0, my0, mx1, my1 = mineru_bbox
            fx0, fy0, fx1, fy1 = mx0 * page_width, my0 * page_height, mx1 * page_width, my1 * page_height
            
            total_chars, italic_chars, bold_chars = 0, 0, 0
            font_sizes = []
            first_line_x0 = None
            
            for block in page_dict.get("blocks", []):
                if block.get("type") != 0: continue
                for line in block.get("lines", []):
                    line_bbox = line.get("bbox")
                    if line_bbox[2] < fx0 or line_bbox[0] > fx1 or line_bbox[3] < fy0 or line_bbox[1] > fy1:
                        continue
                    ix0, iy0 = max(fx0, line_bbox[0]), max(fy0, line_bbox[1])
                    ix1, iy1 = min(fx1, line_bbox[2]), min(fy1, line_bbox[3])
                    inter_area = max(0, ix1 - ix0) * max(0, iy1 - iy0)
                    line_area = (line_bbox[2] - line_bbox[0]) * (line_bbox[3] - line_bbox[1])
                    if line_area == 0 or inter_area / line_area < 0.3: continue
                    if first_line_x0 is None: first_line_x0 = line_bbox[0]
                        
                    for span in line.get("spans", []):
                        text = span.get("text", "").strip()
                        if not text: continue
                        chars = len(text)
                        flags = span.get("flags", 0)
                        font = span.get("font", "").lower()
                        is_italic = bool(flags & 2) or "italic" in font
                        is_bold = bool(flags & 16) or "bold" in font
                        total_chars += chars
                        if is_italic: italic_chars += chars
                        if is_bold: bold_chars += chars
                        font_sizes.append((span.get("size", 11), chars))
            
            style = []
            if total_chars > 0:
                if bold_chars / total_chars > 0.5: style.append("bold")
                if italic_chars / total_chars > 0.5: style.append("italic")
            avg_size = sum(sz * ch for sz, ch in font_sizes) / total_chars if total_chars > 0 else 11
            return {"style": style, "font_size": avg_size, "x0": first_line_x0}

        for i, page in enumerate(doc):
            if is_cancelled: break
            print(f"Đang xử lý trang {i+1}...", flush=True)
            img_bytes = page.get_pixmap(dpi=150).tobytes()
            pil_image = Image.open(io.BytesIO(img_bytes))
            
            result = global_client.predict(pil_image)
            
            for item in result:
                if item["type"] in ["image", "table", "formula"]:
                    bbox = item["bbox"]
                    x0, y0, x1, y1 = bbox
                    w, h = pil_image.width, pil_image.height
                    crop_box = (x0 * w, y0 * h, x1 * w, y1 * h)
                    cropped_img = pil_image.crop(crop_box)
                    buffered = io.BytesIO()
                    cropped_img.save(buffered, format="PNG")
                    img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")
                    item["image_base64"] = img_str
                
                if item["type"] in ["text", "title"]:
                    style_info = extract_style_from_fitz(item["bbox"], page)
                    item.update(style_info)
                    
                item["page_idx"] = i
                item["text"] = item.get("text", "").strip()
                full_data.append(item)
                
        doc.close()
        os.remove(local_path)
        
        return {"status": "success", "data": full_data}
        
    except Exception as e:
        traceback.print_exc()
        return {"status": "error", "message": str(e)}
"""

with open("app.py", "w", encoding="utf-8") as f:
    f.write(server_code)
print("Đã tạo file app.py - Chứa API Server")


# 3. Khởi động Server qua Ngrok
Điền Token Ngrok của bạn vào ô dưới, nếu có tên miền tĩnh (Static Domain) thì điền vào. Nếu không điền Static Domain, Ngrok sẽ random link.

In [ ]:
from pyngrok import ngrok
import nest_asyncio
import uvicorn
import multiprocessing
import time

# =====================================================
# ĐIỀN THÔNG TIN NGROK CỦA BẠN VÀO ĐÂY
NGROK_TOKEN = "2wZkQViTDUMaEHlofUSIbgxbsC6_52ZFY5ERa2LYycbfmSR9j"

# Nếu bạn đã claim Static Domain (vd: xxx.ngrok-free.app), hãy điền vào.
# Nếu để trống, Ngrok sẽ tự random link mới mỗi lần chạy.
NGROK_STATIC_DOMAIN = "advanced-eagerly-lioness.ngrok-free.app"
# =====================================================

nest_asyncio.apply()

def run_server():
    uvicorn.run("app:app", host="127.0.0.1", port=48921, log_level="info")

# Chạy FastAPI ngầm
server_process = multiprocessing.Process(target=run_server)
server_process.start()
time.sleep(3)

if NGROK_TOKEN == "ĐIỀN_TOKEN_NGROK_CỦA_BẠN_VÀO_ĐÂY":
    print("\n⚠️ LỖI: BẠN CHƯA ĐIỀN NGROK TOKEN VÀO CODE CỦA COLAB! ⚠️\n")
else:
    # Bật Ngrok tunnel
    ngrok.set_auth_token(NGROK_TOKEN)
    try:
        if NGROK_STATIC_DOMAIN:
            public_url = ngrok.connect(48921, domain=NGROK_STATIC_DOMAIN).public_url
        else:
            public_url = ngrok.connect(48921).public_url

        print("\n" + "="*60)
        print(f" 🚀 ĐƯỜNG LINK NGROK CỦA BẠN: {public_url} 🚀 ")
        print(" COPY VÀ DÁN VÀO DÒNG 13 CỦA server.py ")
        print("="*60 + "\n")
    except Exception as e:
        print(f"Lỗi kết nối Ngrok: {e}")
